## AI611 Final Project

# DreamerV3 Beyond Benchmarks: Cross-Domain Evaluation of World Model

Team 11: Seungyeon Ryu, Hyeonseo Yun, Seungmin Cha, Kihyun Seol 

## 1. Introduction

Recent advances in reinforcement learning have demonstrated remarkable performance across a wide range of domains. However, many algorithms require extensive task-specific tuning and often struggle to generalize across diverse environments. DreamerV3 addresses this challenge by introducing a robust world-model-based reinforcement learning framework that can achieve strong performance across more than 150 tasks using a single set of hyperparameters.

Motivated by these results, this project investigates the capabilities of DreamerV3 from two perspectives. First, we conduct a **comparative study of DreamerV2 and DreamerV3** using the Atari and Minecraft benchmarks to analyze the improvements introduced in the newer version. Second, we evaluate the **generalization ability of DreamerV3** beyond the environments considered in the original paper. To this end, we train and evaluate DreamerV3 on Highway-Env, a driving simulation benchmark that is not included in the original DreamerV3 task suite, and examine whether the algorithm can successfully adapt to this new domain.



## 2. DreamerV2 vs. DreamerV3: Performance Comparison

### 2.1 Enviroment Setup

Run these steps in a **fresh local clone** (no `/mnt/...` assumptions):

1. Clone branch `dreamerv2-v3` and create Python 3.11 environment.
2. Install deps: `pip install -U -r requirements.txt huggingface_hub gymnasium highway-env imageio ruamel.yaml`.
3. (Optional) set HF token: `export HF_TOKEN=...` for private/quota-safe downloads.
4. Open this notebook at repo root and run cells top-to-bottom.
5. Default mode uses cached artifacts when present; set env flags only when needed:
   - `RUN_MC_LIVE=1` for live Minecraft rollout
   - `REFRESH_ATARI_GIFS=1` for 3-game Atari GIF refresh
   - `RUN_ATARI_LIVE=1` for timed retraining pipeline
6. Highway checkpoint is fetched from HF repo `HyunseoYun/dreamerv3-custom-envs` automatically when missing.

In [ ]:
import io, os, pathlib, sys, urllib.request, zipfile
from IPython.display import Image, display, Markdown
%matplotlib inline

REPO, BRANCH = 'franktome/Dreamerv3_RL_project', 'dreamerv2-v3'
WORKSPACE = pathlib.Path('.').resolve()
if pathlib.Path('dvbench/__init__.py').exists():
    sys.path.insert(0, str(WORKSPACE))
else:
    cache = pathlib.Path.home() / '.cache' / 'dvbench_pkg'
    cache.mkdir(parents=True, exist_ok=True)
    url = f'https://github.com/{REPO}/archive/refs/heads/{BRANCH}.zip'
    with urllib.request.urlopen(url, timeout=180) as r:
        data = r.read()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        zf.extractall(cache)
    sys.path.insert(0, str(next(cache.glob(f'*-{BRANCH}'))))

from dvbench.paths import default_paths
from dvbench.env_setup import clone_dreamerv3, setup_jax, ensure_xvfb
from dvbench.hf_assets import login_if_needed, resolve_minecraft_logdir
from dvbench import inference_demo
from dvbench import viz_advanced

login_if_needed()
cfg = default_paths(WORKSPACE, gpu='1')
clone_dreamerv3(cfg)
setup_jax(cfg)
MC_LOGDIR = resolve_minecraft_logdir(cfg)
print('Minecraft logdir:', MC_LOGDIR)

### 2.2 Atari Results

Comparative evaluation of **DreamerV2** (TensorFlow) and **DreamerV3** (JAX) on **pong**, **breakout**, and **boxing**. Code loads from GitHub branch `dreamerv2-v3` (§2.1); checkpoints from [Hugging Face `checkpoints/atari_*`](https://huggingface.co/HyunseoYun/dreamerv3-custom-envs/tree/main/checkpoints).



#### 2.2.1 Training protocol

All three Atari tasks were trained with DreamerV2 and DreamerV3 under one shared protocol. Six runs in total (three games × two models) used the config stack **`atari` + `atari_compare`**: action repeat 4, sticky actions 0.25, 30 no-op reset steps, grayscale observations, tanh reward clipping, batch size 4 with sequence length 32, and matching RSSM and optimizer settings (discount 0.999, KL scale 0.1, learning rates 2e-4 / 4e-5 / 1e-4). Each game–model pair was trained for **500,000 environment steps** (post action-repeat interactions; ~2M logged frames at repeat 4).



In [ ]:
import json, shutil, subprocess
from pathlib import Path
import pandas as pd

from dvbench.hf_assets import HF_REPO, login_if_needed
from dvbench import atari_compare, atari_align, gif_compare, viz_advanced, viz
from huggingface_hub import snapshot_download

GAMES = atari_compare.GAMES


def ensure_dreamerv2(cfg):
    train_py = cfg.dreamerv2_root / 'dreamerv2' / 'train.py'
    if train_py.exists():
        return
    cfg.dreamerv2_root.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/danijar/dreamerv2.git', str(cfg.dreamerv2_root)],
        check=True,
    )


def _copy_tree(src: Path, dst: Path):
    if not src.exists():
        return
    for p in src.rglob('*'):
        if not p.is_file():
            continue
        out = dst / p.relative_to(src)
        out.parent.mkdir(parents=True, exist_ok=True)
        if not out.exists() or p.stat().st_mtime > out.stat().st_mtime:
            shutil.copy2(p, out)


def ensure_hf_checkpoints(cfg):
    login_if_needed()
    cache = cfg.workspace / 'dreamerv3_checkpoints'
    marker = cache / 'checkpoints' / 'atari_pong_v2' / 'variables.pkl'
    patterns = ['checkpoints/minecraft_diamond_full/**']
    for game in GAMES:
        patterns += [
            f'checkpoints/atari_{game}_v2/**',
            f'checkpoints/atari_{game}_v3/**',
        ]
    if not marker.exists():
        print(f'Downloading {HF_REPO} checkpoints …')
        snapshot_download(HF_REPO, local_dir=str(cache), repo_type='model', allow_patterns=patterns)
    for game in GAMES:
        for ver in ('v2', 'v3'):
            _copy_tree(cache / 'checkpoints' / f'atari_{game}_{ver}', cfg.atari_logdir(game, ver))
    _copy_tree(cache / 'checkpoints' / 'minecraft_diamond_full', cfg.minecraft_full_logdir)

ensure_dreamerv2(cfg)
ensure_hf_checkpoints(cfg)
cfg.apply_env(mem_fraction=0.25)
display(Markdown(f"**Games:** {', '.join(GAMES)} | checkpoints: `{HF_REPO}`"))



#### 2.2.2 Official Atari benchmark (57 games)

We use **Human Normalized Score (HNS)** to compare Atari agents on a common scale. For each game $g$, let $S$ be the agent score, $H_g$ the **human gamer** baseline, and $R_g$ the **random** baseline (from `baselines.json`):

$$
\mathrm{HNS}_g(S)=\frac{S-R_g}{H_g-R_g}
$$

- $\mathrm{HNS}_g=0$: random-level performance
- $\mathrm{HNS}_g=1$: human-level performance
- $\mathrm{HNS}_g>1$: above human

**How it appears in the plots below:**

1. **Per-game learning curves** — at each training budget $x$, the published score $S_x$ is converted via the formula above.
2. **Atari 57 median curve** — at each $x$, take the median over all games:
   $$
   \mathrm{MedianHNS}(x)=\mathrm{median}_{g}\,\mathrm{HNS}_g\!\left(S_{g,x}\right)
   $$
3. **Bar charts / win rate** — use $\mathrm{HNS}_g$ at the **50M-step** budget (last score with $x\le 50{,}000{,}000$).


In [ ]:
compare = viz_advanced.load_atari_compare(cfg)
games10 = viz_advanced.select_10_games(compare)
for plot_fn, out_name in (
    (viz.plot_atari_v2_v3, 'atari_v2_v3.png'),
    (viz_advanced.plot_atari_10game_bars, None),
    (viz_advanced.plot_atari_10game_curves, None),
):
    if plot_fn is viz.plot_atari_v2_v3:
        plot_fn(cfg, show=False)
        p = cfg.report_dir / out_name
    elif plot_fn is viz_advanced.plot_atari_10game_bars:
        p = Path(viz_advanced.plot_atari_10game_bars(cfg, games10, compare, show=False))
    else:
        p = Path(viz_advanced.plot_atari_10game_curves(cfg, games10, show=False))
    if p.exists():
        display(Image(filename=str(p)))

display(Markdown(
    f"**Median HNS:** V2={compare['DreamerV2'].median():.2f} → V3={compare['DreamerV3'].median():.2f} "
    f"({(compare['delta'] > 0).mean() * 100:.0f}% games favor V3)"
))
top = compare.sort_values('Δ', ascending=False).head(5)[['DreamerV2', 'DreamerV3', 'Δ']]
top.index = [i.replace('atari_', '') for i in top.index]
display(top)



##### 2.2.2.1 Analysis

- **Full Atari 57 V2 vs V3** (median curve + per-game bars) summarizes official 50M-step published scores.
- **10-game bars** focus on the largest V3 gains plus two V2-competitive titles.
- **Δ** = DreamerV3 − DreamerV2 HNS @ 50M; positive Δ means V3 wins that game.
- This section uses official score trajectories for reproducibility (no per-game checkpoint rollouts required).


#### 2.2.3 Three-game V2 vs V3 comparison

Matched **500k-step** training on pong / breakout / boxing. Metrics and learning curves use each game's aligned step budget; side-by-side GIFs use checkpoints at or before that step.

| | DreamerV2 | DreamerV3 |
|--|-----------|-----------|
| Backend | TensorFlow | JAX |
| World model | RSSM + imagination | RSSM + imagination (symlog, symexp two-hot rewards, percentile return norm) |

Set `REFRESH_ATARI_GIFS=1` to re-run live inference from HF checkpoints.



In [ ]:
REFRESH = bool(os.environ.get('REFRESH_ATARI_GIFS'))
align_df = pd.DataFrame(atari_align.alignment_table(cfg, GAMES))
display(Markdown('##### 2.2.3.1 Alignment summary'))
show = align_df[['game', 'v2_env_steps', 'v3_env_steps', 'aligned_env_steps', 'fair_gif']].copy()
show.columns = ['game', 'V2 steps', 'V3 steps', 'aligned steps', 'fair GIF']
display(show)

metrics = atari_compare.collect_metrics(cfg)
display(Markdown('##### 2.2.3.2 Training metrics (aligned steps)'))
display(metrics['summary'])

for name in ('learning_curves_3games.png', 'metrics_summary.png'):
    p = cfg.report_dir / 'atari_compare' / name
    if p.exists():
        display(Image(filename=str(p)))

display(Markdown('##### 2.2.3.3 Side-by-side rollouts'))
for game in GAMES:
    row = align_df[align_df['game'] == game].iloc[0]
    gif_path = cfg.highlights_dir / 'inference' / f'atari_{game}_v2v3_compare.gif'
    strip_path = cfg.report_dir / 'atari_compare' / f'{game}_v2v3_strip.png'
    cmp = None
    if REFRESH or (not gif_path.exists() and not strip_path.exists()):
        cmp = gif_compare.infer_both(cfg, game, max_steps=1200, align=True)
    fair = row['fair_gif']
    if cmp and cmp.get('ok'):
        gif_path = Path(cmp.get('compare_gif', gif_path))
        v2s, v3s = cmp.get('v2', {}).get('score'), cmp.get('v3', {}).get('score')
        score_txt = f' | rollout V2={v2s} V3={v3s}' if v2s is not None else ''
    else:
        result_path = cfg.report_dir / 'atari_compare' / 'fair_retrain_result.json'
        if result_path.exists():
            block = json.loads(result_path.read_text()).get('compare', {}).get(game, {})
            v2s = (block.get('v2') or {}).get('score')
            v3s = (block.get('v3') or {}).get('score')
            score_txt = f' | cached V2={v2s} V3={v3s}' if v2s is not None else ''
        else:
            score_txt = ''
    status = 'GIF' if fair and (gif_path.exists() or strip_path.exists()) else 'check alignment'
    display(Markdown(
        f"**{game}** @ **{int(row['aligned_env_steps']):,}** env steps {status}{score_txt}"
    ))
    if gif_path.exists():
        display(Image(filename=str(gif_path)))
    elif strip_path.exists():
        display(Image(filename=str(strip_path)))
    else:
        display(Markdown(f'*No cached GIF for {game}; set `REFRESH_ATARI_GIFS=1`.*'))



##### 2.2.3.4 Analysis

- Alignment table ensures V2 and V3 GIFs use checkpoints at the same **environment-step** budget per game.
- Training metrics and learning curves are truncated to each game's aligned step count.
- In V2|V3 compare GIFs, DreamerV2 rollouts tend to show **low action entropy**, so the policy rarely switches actions. **Frame counts may also differ** between V2 and V3 (e.g. breakout), which can desync the two sides for part of the clip.


#### 2.2.4 Environment perturbations

Same **trained checkpoints** (§2.2.1) are rolled out under perturbed env wrappers. Use the **interactive explorer** below to switch among cached presets (`baseline` / `fast` / `sluggish`). Sliders snap to the nearest preset — no live re-inference unless `REFRESH_ATARI_GIFS=1`.


In [ ]:
from dvbench import atari_anim_ui

try:
    atari_anim_ui.show_perturb_explorer(cfg, GAMES)
except Exception as exc:
    display(Markdown(f'*Interactive explorer unavailable ({exc}).*'))


##### 2.2.4.1 Analysis

### Parameters

| Parameter | Meaning | Training value | Explorer range | If you **increase** | If you **decrease** |
|-----------|---------|---------------:|----------------|---------------------|---------------------|
| **`repeat`** | ALE frames per agent decision (frame skip) | 4 | slider **2–6**; cached presets **2** or **4** | Fewer decisions per game second → coarser control | More frequent decisions → finer control |
| **`sticky`** | Probability ALE repeats the **previous** action (`repeat_action_probability`) | 0.25 | slider **0.00–0.75**; cached **0.25** or **0.50** | More input lag → actions “stick” | Crisper response to policy output |
| **`noops`** | Random NOOP actions after `reset` | 30 | fixed **30** (not in UI) | Longer random idle after reset | Shorter idle → more deterministic starts |

### Cached presets

| Preset | repeat | sticky | noops | vs training |
|--------|-------:|-------:|------:|-------------|
| `baseline` | 4 | 0.25 | 30 | matches §2.2.1 training env |
| `fast` | 2 | 0.25 | 30 | lower frame skip |
| `sluggish` | 4 | 0.50 | 30 | higher action stickiness |

### Rollout returns (aligned checkpoints)

Scores are episode return from `infer_both(..., align=True)`; Δ = perturbed − baseline on the same model.

| Game | Preset | V2 | V3 | ΔV2 | ΔV3 |
|------|--------|---:|---:|----:|----:|
| pong | baseline | −16 | −4 | 0 | 0 |
| pong | fast | −7 | −3 | **+9** | **+1** |
| pong | sluggish | −16 | −8 | 0 | **−4** |
| breakout | baseline | 0 | 7 | 0 | 0 |
| breakout | fast | 0 | 6 | 0 | −1 |
| breakout | sluggish | 0 | 11 | 0 | **+4** |
| boxing | baseline | 17 | 12 | 0 | 0 |
| boxing | fast | 2 | −16 | **−15** | **−28** |
| boxing | sluggish | 10 | 6 | −7 | −6 |

**Patterns:** `fast` (repeat↓) helped pong slightly but hurt boxing strongly; `sluggish` (sticky↑) hurt pong V3 and boxing both models, while breakout V3 improved (+4).

- **fast** (`repeat=2`): shorter frame skip → faster ball/paddle dynamics.
- **sluggish** (`sticky=0.5`): actions repeat more often → delayed response.
- Use the explorer to compare how **the same checkpoint** behaves under each cached perturbation.
- Sliders map to the nearest preset; set `REFRESH_ATARI_GIFS=1` to refresh GIFs and `anim_index.json` scores.


### 2.3 Minecraft Results

DreamerV3 on **MineRL diamond** (`minecraft_diamond`). Official V3 / PPO / IMPALA curves use published scores; local rollout uses the HF checkpoint `checkpoints/minecraft_diamond_full`.



#### 2.3.1 Official Minecraft benchmark

**Milestone episode score** (MineRL `minecraft_diamond`). Each of 12 items grants a one-time sparse reward the first time it appears in inventory:

$$r_j = \mathbb{1}\big[\text{first\_collect}(m_j)\big], \quad j=1,\ldots,12$$

Episode return (logged as `ys` in official JSON):

$$R = \sum_{j=1}^{12} r_j + \varepsilon_{\text{health}} \in [0,12]$$

**Diamond completion** requires $R \ge 12$.

**Left plot — Task success (%)**

$$\text{Success}_s(t)=\mathbb{1}\big[Y_s(t)\ge 12\big], \qquad \text{TaskSuccess}(t)=\frac{100}{N}\sum_{s=1}^{N}\text{Success}_s(t)$$

**Right plot — Mean max milestone**

$$\bar{M}(t)=\frac{1}{N}\sum_{s=1}^{N}\min\!\big(12,\;Y_s(t)\big)$$

DreamerV3 reaches **~70%** diamond success vs **0%** for PPO/IMPALA on official JSON, while PPO/IMPALA still reach mean depth $\approx$10–11 without completing the final milestone.


In [ ]:
ensure_hf_checkpoints(cfg)
MC_LOGDIR = cfg.minecraft_full_logdir

# Summary only — no plot display (figures may still be cached under report/)
mc_summary = viz_advanced.plot_minecraft_baselines_enhanced(cfg, show=False)
_v3 = mc_summary.get('DreamerV3', {})
display(Markdown(
    f"Official summary — V3 task success **{_v3.get('final_task_success_pct', 0):.0f}%**, "
    f"mean milestone **{_v3.get('mean_max_milestone', 0):.2f}**"
))


##### 2.3.1.1 Analysis

| Method | Seeds $N$ | TaskSuccess($\infty$) | Final $\bar{M}$ |
|--------|----------:|------------------------:|----------------:|
| DreamerV3 | 20 | **70%** (14/20) | **11.95** |
| PPO | 14 | **0%** (0/14) | **10.92** |
| IMPALA | 15 | **0%** (0/15) | **11.00** |

**Why left ≈ 0% but right looks similar?**

| | Left: TaskSuccess | Right: $\bar{M}$ |
|--|-------------------|-------------------|
| Type | **binary** ($R\ge12$?) | **continuous** average depth |
| PPO/IMPALA | fail final milestone (diamond) | still reach depth $\sim$10–11 |
| Visual gap | large (0% vs 70%) | small (10.9–11.0 vs 12) |

**Algorithmic difference:** DreamerV3 uses a **world model** (RSSM) + **imagination** for multi-step craft planning; PPO/IMPALA are **model-free** and master the early chain (high $\bar{M}$) but rarely complete the sparse diamond step.

- **Atari (§2.2):** V3 improves median HNS and wins most per-game comparisons.
- **Minecraft (plots):** same official JSON as DreamerV3 repo; left = completion rate, right = mean depth — complementary, not contradictory.


#### 2.3.2 Policy rollout

Rollout from the HF checkpoint. Default: cached GIFs under `highlights/inference/`; set `RUN_MC_LIVE=1` for a fresh env rollout.



In [ ]:
from dvbench.inference_demo import (
    analyze_training_health,
    display_minecraft_inference,
    load_minecraft_rollout_index,
    run_minecraft_multi_rollouts,
)

ensure_hf_checkpoints(cfg)
MC_LOGDIR = cfg.minecraft_full_logdir
RUN_LIVE = os.environ.get('RUN_MC_LIVE')
RUN_MULTI = os.environ.get('RUN_MC_MULTI')

if RUN_MULTI and (MC_LOGDIR / 'ckpt').exists():
    ensure_xvfb(cfg.display)
    cfg.apply_env(mem_fraction=0.42)
    mc_index = run_minecraft_multi_rollouts(cfg, logdir=MC_LOGDIR, n_rollouts=8, top_k=3, max_steps=3600)
elif RUN_LIVE and (MC_LOGDIR / 'ckpt').exists():
    ensure_xvfb(cfg.display)
    live = inference_demo.run_minecraft_env_gif(cfg, logdir=MC_LOGDIR, max_steps=3600)
    mc_index = {'ok': live.get('ok'), 'best': live, 'top_k': [{**live, 'rank': 1}],
                'checkpoint_id': live.get('checkpoint_id'), 'training_step': live.get('training_step')}
else:
    mc_index = load_minecraft_rollout_index(cfg, logdir=MC_LOGDIR, top_k=3)

health = analyze_training_health(MC_LOGDIR) if (MC_LOGDIR / 'ckpt').exists() else {'episodes': 0}
mc_result = display_minecraft_inference(cfg, mc_index, health, logdir=MC_LOGDIR)



##### 2.3.2.1 Analysis

- Episode **reward** reflects cumulative milestone index during the rollout.
- **Milestone strip** labels each inventory unlock; wider strip = more progress within the episode window.
- Compare with §2.3.1 official curves to see where this checkpoint sits relative to published V3 runs.

<small>

**Checkpoint & inference.** Rollout uses the HF checkpoint (`ckpt/latest`). A single inference rollout can show fewer events than the best training episode.

**Why diamond is unrealistic on a short local run.** Official DreamerV3 reaches diamond at a **median ~52M steps**. Short local training runs typically stall near **crafting_table / wooden_pickaxe**; finishing diamond on one GPU is not a practical inference-demo goal.

</small>


### 2.4 Discussion

**Atari.** DreamerV3 improves median HNS on the official 57-game suite and wins most head-to-head comparisons on the 10-game subset. On our three matched 500k-step runs (pong, breakout, boxing), V3 matches or exceeds V2 on aligned training metrics; environment perturbations show V3 policies can be more or less robust depending on game and perturbation type.

**Minecraft.** On official benchmarks DreamerV3 is the only method that reliably discovers diamond (~70% seed success); PPO and IMPALA reach similar mean milestone depth but fail the final sparse milestone. Local checkpoint rollouts illustrate the multi-step craft chain learned by the world model.

**Takeaway.** DreamerV3's symlog inputs, distributional reward/value heads, and return normalization support a single hyperparameter set across Atari and Minecraft, while matched-step three-game training and perturbation probes highlight where V2 and V3 differ in sample efficiency and dynamics robustness.



## 3. Generalization of DreamerV3 on Highway-Env

We apply DreamerV3 to Highway-Env, a driving simulation benchmark focused on autonomous navigation and decision-making.

Highway-Env provides a variety of traffic scenarios, including highway driving, merging, roundabout navigation, and intersection crossing. These tasks require the agent to balance safety, efficiency, and long-term planning while interacting with other vehicles. We selected Highway-Env because it represents a domain that is substantially different from the environments used in the original DreamerV3 benchmark suite, while still offering a diverse set of tasks within a unified framework.

In this section, we train DreamerV3 on several Highway-Env tasks and evaluate whether the algorithm can successfully learn effective driving behaviors in previously unseen environments.

### 3.1 Environment Setup (Reproducibility Guide)

This notebook was tested using the software environment specified in `environment.yaml`.

Before running the notebook:

1. Download the environment.yaml file.

```bash
wget https://raw.githubusercontent.com/franktome/Dreamerv3_RL_project/dreamerv2-v3/environment.yaml
```

2. Create the conda environment.

```bash
conda env create -f environment.yaml
```

3. Activate the environment.

```bash
conda activate dreamerv3
```

4. Run all cells from top to bottom.

DreamerV3 Source Code
- This notebook automatically downloads the DreamerV3 source code if it is not already present.

Checkpoint Download
- Pretrained checkpoints are automatically downloaded from the Hugging Face repository. No manual checkpoint download is required.



In [ ]:
config_dict = {
    "loss_scales": {
        "rec": 1.0, "rew": 1.0, "con": 1.0, "dyn": 1.0,
        "rep": 0.1, "policy": 1.0, "value": 1.0, "repval": 0.3
    },
    "opt": {
        "lr": 4e-05, "agc": 0.3, "eps": 1e-20, "beta1": 0.9,
        "beta2": 0.999, "momentum": True, "wd": 0.0,
        "schedule": "const", "warmup": 1000, "anneal": 0
    },
    "ac_grads": False,
    "dyn": {
        "typ": "rssm",
        "rssm": {
            "deter": 2048, "hidden": 256, "stoch": 32, "classes": 16,
            "act": "silu", "norm": "rms", "unimix": 0.01,
            "outscale": 1.0, "winit": "trunc_normal_in",
            "imglayers": 2, "obslayers": 1, "dynlayers": 1,
            "absolute": False, "blocks": 8, "free_nats": 1.0
        }
    },
    "enc": {
        "typ": "simple",
        "simple": {
            "depth": 16, "mults": [2, 3, 4, 4], "layers": 3,
            "units": 256, "act": "silu", "norm": "rms",
            "winit": "trunc_normal_in", "symlog": True,
            "outer": False, "kernel": 5, "strided": False
        }
    },
    "dec": {
        "typ": "simple",
        "simple": {
            "depth": 16, "mults": [2, 3, 4, 4], "layers": 3,
            "units": 256, "act": "silu", "norm": "rms",
            "outscale": 1.0, "winit": "trunc_normal_in",
            "outer": False, "kernel": 5, "bspace": 8, "strided": False
        }
    },
    "rewhead": {
        "layers": 1, "units": 256, "act": "silu", "norm": "rms",
        "output": "symexp_twohot", "outscale": 0.0,
        "winit": "trunc_normal_in", "bins": 255
    },
    "conhead": {
        "layers": 1, "units": 256, "act": "silu", "norm": "rms",
        "output": "binary", "outscale": 1.0, "winit": "trunc_normal_in"
    },
    "policy": {
        "layers": 3, "units": 256, "act": "silu", "norm": "rms",
        "minstd": 0.1, "maxstd": 1.0, "outscale": 0.01,
        "unimix": 0.01, "winit": "trunc_normal_in"
    },
    "value": {
        "layers": 3, "units": 256, "act": "silu", "norm": "rms",
        "output": "symexp_twohot", "outscale": 0.0,
        "winit": "trunc_normal_in", "bins": 255
    },
    "policy_dist_disc": "categorical",
    "policy_dist_cont": "bounded_normal",
    "imag_last": 0,
    "imag_length": 15,
    "horizon": 333,
    "contdisc": True,
    "imag_loss": {"slowtar": False, "lam": 0.95, "actent": 0.0003, "slowreg": 1.0},
    "repl_loss": {"slowtar": False, "lam": 0.95, "slowreg": 1.0},
    "slowvalue": {"rate": 0.02, "every": 1},
    "retnorm": {"impl": "perc", "rate": 0.01, "limit": 1.0, "perclo": 5.0, "perchi": 95.0, "debias": False},
    "valnorm": {"impl": "none", "rate": 0.01, "limit": 1e-08},
    "advnorm": {"impl": "none", "rate": 0.01, "limit": 1e-08},
    "reward_grad": True,
    "repval_loss": True,
    "repval_grad": True,
    "report": True,
    "report_gradnorms": False,
    "seed": 0,
    "logdir": "",
    "jax": {
        "platform": "cuda", "compute_dtype": "bfloat16",
        "policy_devices": [0], "train_devices": [0],
        "mock_devices": 0, "prealloc": True, "jit": True,
        "debug": False, "expect_devices": 0, "enable_policy": True,
        "coordinator_address": ""
    },
    "batch_size": 16,
    "batch_length": 64,
    "replay_context": 1,
    "report_length": 32,
    "replica": 0,
    "replicas": 1
}

tasks = [
    {
        "env_id": "highway-v0", 
        "ckpt_path": "checkpoints/highway_highway", 
        "duration": 100,
        "gif_name": "highway.gif"
    },
    {
        "env_id": "highway-fast-v0", 
        "ckpt_path": "checkpoints/highway_train_v1", 
        "duration": 100,
        "gif_name": "highway_fast.gif"
    },
    {
        "env_id": "roundabout-v0", 
        "ckpt_path": "checkpoints/highway_roundabout", 
        "duration": 100,
        "gif_name": "roundabout.gif"
    },
    {
        "env_id": "intersection-v0", 
        "ckpt_path": "checkpoints/highway_intersection", 
        "duration": 100,
        "gif_name": "intersection.gif"
    },
    {
        "env_id": "merge-v0", 
        "ckpt_path": "checkpoints/highway_merge", 
        "duration": 100,
        "gif_name": "merge.gif"
    },
]

In [ ]:
#!pip install highway-env
import os
import contextlib
import sys
import pathlib
import warnings
import imageio
import numpy as np
import gymnasium as gym
import highway_env
import elements
from huggingface_hub import snapshot_download
from IPython.display import Image, display

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["SDL_AUDIODRIVER"] = "dummy"

warnings.filterwarnings("ignore")

if not os.path.exists("dreamerv3"):
    !git clone https://github.com/danijar/dreamerv3.git
    #!git clone -b dreamerv2-v3 https://github.com/franktome/Dreamerv3_RL_project.git|

sys.path.insert(0, "./dreamerv3")

from dreamerv3.agent import Agent

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['JAX_PLATFORMS'] = 'cuda'

import jax
print('JAX devices:', jax.devices())

REPO_ID = "HyunseoYun/dreamerv3-custom-envs"
DOWNLOAD_DIR = "./dreamerv3_checkpoints"

if not os.path.exists(DOWNLOAD_DIR):
    print("Checkpoint downloading ...")
    snapshot_download(repo_id=REPO_ID, local_dir=DOWNLOAD_DIR)


### 3.2 Inference

After training, we evaluated DreamerV3 using pretrained checkpoints obtained from the Highway-Env tasks. For each task, the corresponding checkpoint was loaded and the agent was executed in evaluation mode without further learning or parameter updates.

At the beginning of each episode, the environment was initialized and the trained policy interacted with the environment by selecting actions based on the current observation. The evaluation continued until the episode terminated or reached the predefined time limit. During execution, rendered frames from the environment were collected and saved as GIF files to visualize the agent's behavior.

The inference process was conducted on several Highway-Env scenarios, including highway driving, high-speed highway driving, roundabout navigation, intersection crossing, and merging. These scenarios cover different traffic conditions and decision-making requirements, allowing us to examine whether the learned policy can generalize across a variety of driving situations.

The generated GIFs provide a qualitative assessment of the learned behaviors, while the cumulative episode rewards serve as a quantitative measure of task performance.

*Note*:
The complete inference process across all Highway-Env tasks required approximately 12 minutes on our experimental setup. This includes loading pretrained checkpoints, initializing the DreamerV3 world model, executing policy rollouts, rendering environment frames, and generating GIF visualizations.


In [ ]:
for task in tasks:
    env_id = task["env_id"]

    env = gym.make(env_id, render_mode='rgb_array', config={"duration": task["duration"]})

    obs_space = {
        "obs": elements.Space(np.float32, env.observation_space.shape),
        "reward": elements.Space(np.float32),
        "is_first": elements.Space(bool),
        "is_last": elements.Space(bool),
        "is_terminal": elements.Space(bool),
    }
    act_space = {"action": elements.Space(np.int32, (), 0, env.action_space.n)}

    current_config = elements.Config(config_dict).update(logdir=f"logdir/{env_id}")

    target_ckpt_dir = pathlib.Path(DOWNLOAD_DIR) / task["ckpt_path"]

    try:
        with open(os.devnull, "w") as f:
            with contextlib.redirect_stdout(f):

                agent = Agent(obs_space, act_space, current_config)
                cp = elements.Checkpoint(target_ckpt_dir)
                cp.agent = agent
        cp.load()
        print(f"{env_id} load success : {target_ckpt_dir}")
    except Exception as e:
        print(f"{env_id} load fail : {e}")
        env.close()
        continue

    frames = []
    obs_raw, _ = env.reset()
    done = False
    total_reward = 0
    carry = agent.init_policy(1)
    is_first = True

    while not done:
        frame = env.render()
        frames.append(frame)

        obs = {
            "obs": np.array([obs_raw], dtype=np.float32),
            "reward": np.array([0.0], dtype=np.float32),
            "is_first": np.array([is_first]),
            "is_last": np.array([False]),
            "is_terminal": np.array([False]),
        }
        is_first = False

        carry, act, _ = agent.policy(carry, obs, mode='eval')
        action = int(act['action'][0])
        obs_raw, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        done = terminated or truncated

    env.close()
    print(f"total reward: {total_reward:.2f}, frame: {len(frames)}")

    imageio.mimsave(task["gif_name"], frames, fps=10)
    print("GIF saved successfully!")

### 3.3 Results

The following GIFs show the behaviors learned by DreamerV3 across several Highway-Env tasks. Despite not being part of the original DreamerV3 benchmark suite, the agent successfully learns task-specific driving strategies in a variety of traffic scenarios.

In the highway driving task, the agent maintains stable lane-following behavior while avoiding collisions and making efficient progress. In more challenging scenarios such as roundabouts, intersections, and merging, the agent learns to coordinate with surrounding vehicles, select appropriate gaps in traffic, and execute safe maneuvers. These behaviors indicate that the learned world model can capture the dynamics of complex traffic environments and support effective long-term decision making.

Overall, the results demonstrate that **DreamerV3 can be successfully applied to previously unseen domains and can learn robust driving policies without requiring task-specific modifications**.


In [ ]:
from PIL import Image as PILImage
from IPython.display import Image, display

def make_gif_infinite(gif_path):
    img = PILImage.open(gif_path)
    
    frames = []
    try:
        while True:
            frames.append(img.copy())
            img.seek(len(frames))
    except EOFError:
        pass
    
    frames[0].save(
        gif_path,
        save_all=True,
        append_images=frames[1:],
        duration=img.info.get('duration', 50),
        loop=0
    )

for task in tasks:
    make_gif_infinite(task["gif_name"])
    print(f"{task['env_id']} 환경의 시뮬레이션 결과:")
    display(Image(task["gif_name"]))

### 4. Discussion

### 5. References

1. Hafner, D., Pasukonis, J., Ba, J., & Lillicrap, T. (2023). *Mastering Diverse Domains through World Models (DreamerV3).* arXiv:2301.04104.

2. Ha, D., & Schmidhuber, J. (2018). *World Models.* arXiv:1803.10122. 

3. Hafner, D., Lillicrap, T., Ba, J., & Norouzi, M. (2019). *Learning Latent Dynamics for Planning from Pixels.* Proceedings of the 36th International Conference on Machine Learning (ICML).

4. Hafner, D., Lillicrap, T., Norouzi, M., & Ba, J. (2021). *Mastering Atari with Discrete World Models.* International Conference on Learning Representations (ICLR).